# 02 - Test de généralisation par catégories non vues

Ce notebook teste si le modèle FreshOrRotten généralise à des produits absents de l'entraînement.

Le protocole est différent du `standard_split` :

- certaines catégories sont retirées du train ;
- le modèle est entraîné seulement sur les catégories vues ;
- le test est fait uniquement sur les catégories non vues.

Le but n'est pas forcément d'obtenir une meilleure performance.
Le but est de comprendre les limites de généralisation du modèle.

## Préparation

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

current_path = Path.cwd()
project_root = current_path if (current_path / "config.yaml").exists() else current_path.parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from train import load_config, scan_image_files
from split_strategy import create_unseen_category_split

In [3]:
config = load_config(project_root / "config.yaml")

raw_data_dir = Path(config["paths"]["raw_data_dir"])
dataset_path = raw_data_dir if raw_data_dir.is_absolute() else project_root / raw_data_dir
reports_dir = project_root / Path(config["paths"]["results_report"]).parent
train_script = (project_root / "src" / "train.py").as_posix()
evaluate_script = (project_root / "src" / "evaluate.py").as_posix()

print(f"Project root: {project_root}")
print(f"Dataset path: {dataset_path}")
print(f"Reports dir: {reports_dir}")

Project root: c:\Users\antoc\FreshOrRotten
Dataset path: c:\Users\antoc\FreshOrRotten\data\raw
Reports dir: c:\Users\antoc\FreshOrRotten\reports


## Lecture des images et des catégories

On lit seulement les chemins et les labels.
Les images ne sont pas chargées en mémoire dans cette étape.

In [4]:
image_index_df = scan_image_files(dataset_path, config["data"]["labels"])

category_summary_df = (
    image_index_df.groupby(["product_type", "freshness"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
category_summary_df["total"] = category_summary_df.sum(axis=1)

display(category_summary_df)

freshness,fresh,rotten,total
product_type,,,
apple,3468,4263,7731
banana,3513,3605,7118
bellpepper,1634,2108,3742
bitter_gourd,327,357,684
carrot,605,507,1112
cucumber,833,692,1525
grape,227,288,515
grapes,200,200,400
guava,1352,1329,2681


## Choix des catégories non vues

Choisir ici les catégories à retirer de l'entraînement.
Les noms doivent correspondre à la colonne `product_type` affichée au-dessus.

Exemple : `apple`, `banana`, `tomato`.

In [5]:
# À adapter selon les catégories que vous voulez tester.
unseen_categories = ["apple", "banana", "tomato"]

print("Catégories non vues :", unseen_categories)

Catégories non vues : ['apple', 'banana', 'tomato']


## Vérification du split

`training_set` et `validation_set` contiennent seulement les catégories vues.
`test_set` contient seulement les catégories non vues.

In [6]:
training_set, validation_set, test_set = create_unseen_category_split(
    image_index_df=image_index_df,
    unseen_categories=unseen_categories,
    validation_size=config["training"]["validation_size"],
    random_seed=config["training"]["random_seed"],
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train", "image_count": len(training_set), "product_count": training_set["product_type"].nunique()},
        {"split": "validation", "image_count": len(validation_set), "product_count": validation_set["product_type"].nunique()},
        {"split": "test", "image_count": len(test_set), "product_count": test_set["product_type"].nunique()},
    ]
)

display(split_summary_df)
display(test_set.groupby(["product_type", "freshness"]).size().reset_index(name="image_count"))

,split,image_count,product_count
0,train,27486,19
1,validation,6872,19
2,test,19258,3


,product_type,freshness,image_count
0,apple,fresh,3468
1,apple,rotten,4263
2,banana,fresh,3513
3,banana,rotten,3605
4,tomato,fresh,1905
5,tomato,rotten,2504


## Entraînement avec catégories non vues

Cette cellule réentraîne la même baseline CNN.
Les catégories définies dans `unseen_categories` sont absentes du train.

Sur Colab, il est conseillé d'utiliser un GPU.

In [7]:
unseen_categories_argument = " ".join(unseen_categories)

!python "{train_script}" --split unseen --unseen-categories {unseen_categories_argument}

^C


## Évaluation sur les catégories non vues

In [ ]:
!python "{evaluate_script}" --split unseen --unseen-categories {unseen_categories_argument}

## Résultats globaux

In [ ]:
standard_metrics_path = reports_dir / "evaluation_metrics.csv"
unseen_metrics_path = reports_dir / "unseen_category_evaluation_metrics.csv"
comparison_path = reports_dir / "evaluation_comparison.csv"

if standard_metrics_path.exists():
    standard_metrics_df = pd.read_csv(standard_metrics_path)
    display(standard_metrics_df.assign(protocol="standard_split"))
else:
    print("Les résultats du standard_split sont absents.")

if unseen_metrics_path.exists():
    unseen_metrics_df = pd.read_csv(unseen_metrics_path)
    display(unseen_metrics_df.assign(protocol="unseen_category_split"))
else:
    print("Les résultats du unseen_category_split sont absents.")

if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    display(comparison_df)

## Résultats par catégorie

In [ ]:
category_metrics_path = reports_dir / "unseen_category_metrics_by_product_type.csv"

if category_metrics_path.exists():
    category_metrics_df = pd.read_csv(category_metrics_path)
    display(category_metrics_df)
else:
    print("Les métriques par catégorie sont absentes.")

## Conclusion

Si les performances baissent avec le `unseen_category_split`, ce n'est pas un échec.
C'est une limite intéressante à analyser.

Une baisse peut indiquer que le modèle n'apprend pas seulement des indices généraux de fraîcheur.
Il peut aussi apprendre des indices propres aux catégories vues pendant l'entraînement.

Ce protocole permet donc de mieux comprendre la généralisation du modèle.
Il complète l'accuracy obtenue avec le split classique.